# LLM-augmented BO

**Prepare AI → pick an initial test point → run GNLSE → AI chooses an action → repeat or pause.**

Select **GNLSE (Python)**. Function details stay in the
[action helper](../../../notebook_helpers/llm_bo/actions.py). Section 4 shows the original saved 100 kW comparison.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook_helpers"))
from llm_bo.actions import start_experiment, resume_experiment
from llm_bo.waveforms import compare_saved_waveforms

## 1. Settings

Three zero-phase launch amplitudes satisfy
\[
0\le c_i\le100,\qquad c_1+c_2+c_3=100,\qquad P_i/P=c_i^2/\sum_j c_j^2.
\]
The [reference notebook](../../Modal%20amplitude%20coefficients/Gradient%20descent/dio_three_coefficient_optimization.ipynb) supplies the fiber and pulse.
Total power starts at **100 kW**, with an initial search range of **81.943–100 kW**.
AI may request a recorded power-range expansion; amplitude bounds remain physical limits.

We minimize LP01 D_IO. The initial brightness floor is **50% of the equal-amplitude,
initial-power 1 mm baseline**, and that reference stays fixed when power changes.

`MAX_TESTS` counts every new simulation, including baseline, gradient, and validation work.
The six-test trial allowance is retained. Increase it explicitly for larger actions:
a full gradient check can use **18** new simulations. Cache hits are free.

In [ ]:
POWER_KW = 100.0
POWER_RANGE_KW = (81.943, 100.0)
MAX_TESTS = 6
MAX_AI_CALLS = 4
MAX_DECISIONS = 3
MAX_MINUTES = 10
MODEL = None  # Use the model configured in Codex.
THINKING_EFFORT = None  # Set an effort supported by your model to override.

## 2. Run the search

The short excerpts show the steps inside the run command below.

### 2.1 Prepare AI

Give AI the **context of this run, optimization parameters, and ranges**, using the
selected model and thinking effort. It chooses an initial GP prior for the amplitude
mixture, an initial point at the starting power, and **prints its reasoning briefly**.

\[f(x)\sim\mathcal{GP}\!\left(m(x),k(x,x')\right).\]

The context includes:

```python
{
    "task": "Minimize LP01 D_IO",
    "amplitudes": "0 <= c_i <= 100; c1+c2+c3 = 100; zero phases",
    "power_range_kW": [81.943, 100.0],
    "launch": "200 fs Gaussian; three LP modes",
    "fiber": "Silica step-index; length 2.86 m; carrier 1379.2 nm",
    "brightness": "Initially >= 50% of the fixed initial-power baseline",
}
```

```python
plan = study.prepare_ai()  # Prints the prior, first point, and brief explanation.
```

### 2.2 AI picks an initial test point in the range and starts initiating BO

Python checks the input and measures the 5 mm/1 mm equal-amplitude baseline.

```python
initial_point = plan["initial_point"]
study.initialize()  # Establish the fixed brightness reference.
```

### 2.3 BO does one test point: run a GNLSE and get D_IO

```python
evaluation_id = study.evaluate_initial()  # One input → GNLSE → measured D_IO.
decision = study.choose_next()           # Send saved results to AI.
```

The request inside `choose_next` includes these fields (shortened):

```python
decision = ask({
    "problem": "Minimize unaligned complex LP01 D_IO",
    "observations": history,       # Inputs, D_IO, brightness, resolution, checks.
    "baseline": baseline,          # Fixed reference input.
    "budget": remaining_budget,
    "allowed_actions": list(ACTIONS),
}, decision_schema())
```

### 2.4 AI does the acquisition function's job: Evaluate, update, repeat

Later choices use the measured history; the initial GP is not updated.
Python executes the selected action and sends the resulting measurements back to AI.

```python
result = study.execute(decision)
```

| Allowed action | What it does |
|---|---|
| `evaluate` | Test one input. |
| `descend` | Start or resume gradient descent. |
| `validate_candidate` | Recheck the candidate and baseline at 1 mm. |
| `validate_gradient` | Compare slopes at 5 mm, 1 mm, and Nt = 4096. |
| `expand_range` | Test an outward power probe; expand only if evidence supports it. |
| `robustness` | Test small combined amplitude and power changes. |
| `update_threshold` | Revise a numerical or brightness limit, retaining its history. |
| `pause` | Save a checkpoint and stop; resume later. |

Every decision is printed and saved with these fields:

| Field | Simple meaning |
|---|---|
| `parameters` | Three amplitudes and `power_kW`; `None` when resuming a trajectory. |
| `run_id` | `None` = new search; a saved number = resume that descent trajectory. |
| `evaluations` | Maximum new simulations for this action; cache hits are free. |
| `iterations` | Maximum descent steps or robustness samples. |
| `evidence_ids` | Saved evaluation numbers, e.g. `[3]`. |
| `rationale` | Reason for choosing this action. |
| `knowledge_basis` | The physics idea or hypothesis. |
| `expected_outcome` | What the next test is expected to show. |
| `control` | Variable to expand: `"power_kW"`. |
| `direction` | `"upper"` or `"lower"` bound. |
| `threshold_name`, `threshold_value` | Which numerical/brightness limit to change, and its new value. |

For example (illustrative):

```python
decision = {
    "action": "evaluate",
    "parameters": {"c1": 25, "c2": 37.5, "c3": 37.5, "power_kW": 95},
    "run_id": None, "evaluations": 1, "iterations": 1, "evidence_ids": [3],
    "rationale": "Test a small reduction in power.",
    "knowledge_basis": "Lower power may reduce nonlinear reshaping.",
    "expected_outcome": "Lower D_IO while retaining sufficient brightness.",
    "control": None, "direction": None,
    "threshold_name": None, "threshold_value": None,
}
```

Power expansions use a **25 kW** increment, supported by a refined gradient and probe.
Robustness uses exploratory perturbations of ±0.005 in the LP01 amplitude fraction
and LP11 split, and ±1 kW; these are sensitivity tests, not measured equipment errors.
Threshold changes are versioned. Loosening a tolerance does not improve numerical precision.

**Evaluate → save the result → AI chooses an action → repeat or pause.**

In [ ]:
# File the saved campaign under its launch controls and search method.
from notebook_helpers.run_library import file_saved_run

study = start_experiment(
    power_kW=POWER_KW, power_range_kW=POWER_RANGE_KW,
    max_tests=MAX_TESTS, max_ai_calls=MAX_AI_CALLS, max_minutes=MAX_MINUTES,
    model=MODEL, thinking_effort=THINKING_EFFORT,
)
file_saved_run(study.directory, "Modal amplitudes and total launch power", "LLM-guided actions")
study.run(decisions=MAX_DECISIONS)

## 3. View the results and resume

Each campaign saves its measurements, fields, full decisions, optimizer checkpoints,
range changes, and criteria revisions. The directory is printed when the run starts.

In [ ]:
study.results()

Resume the same campaign in a later session:

```python
study = resume_experiment("output/<saved campaign directory>")
study.run(decisions=3)
```

Remaining budgets and criteria are preserved. If you want to grant additional work,
call `study.allow_more(tests=20, ai_calls=3, minutes=5)` explicitly before continuing.
AI can allocate work within the grant, but cannot increase it.

## Optional waveform comparison at 100 kW

To compare a fixed-power, amplitude-only LLM run with the classical three-coefficient run, provide the LLM run directory and the classical run’s 100 kW case directory.

In [ ]:
FIXED_POWER_LLM_RUN = None
CLASSICAL_CASE = None
if FIXED_POWER_LLM_RUN is not None and CLASSICAL_CASE is not None:
    compare_saved_waveforms(llm_directory=FIXED_POWER_LLM_RUN, previous_case=CLASSICAL_CASE,
                            output_directory=Path(FIXED_POWER_LLM_RUN) / "waveform_comparison")